# 📚 RAGAS & LangChain: End-to-End RAG System Evaluation with Google Gemini

Welcome to the comprehensive masterclass on **RAG System Evaluation** using **RAGAS** (*Retrieval Augmented Generation Assessment*) and **LangChain** powered by **Google Gemini**.

---

### 🎯 Learning Objectives:
1. **Understand RAGAS Metrics Ecosystem**: Learn the standard metric families (RAG, Multimodal, Agent, and NLP comparison metrics).
2. **Master Evaluation Paradigms**: Understand the fundamental differences between **Rubric-based** (qualitative discrete pass/fail) and **Likert-scale** (graded ordinal numeric) evaluation.
3. **Build an End-to-End Grounded RAG Application**: Construct an in-memory retrieval pipeline using `LangChain`, `GoogleGenerativeAIEmbeddings`, `InMemoryVectorStore`, and `ChatGoogleGenerativeAI`.
4. **Implement LLM-as-a-Judge**: Configure an independent evaluator model using Ragas `llm_factory` and Google GenAI.
5. **Execute Systematic Experiments**: Define custom metrics (`DiscreteMetric`, `NumericMetric`), build a golden ground-truth dataset, run asynchronous batch evaluations via `@experiment`, and analyze diagnostic reports.

---

## 1. Ragas Metric Ecosystem Reference
Ragas organizes metrics into distinct collections based on the modality and component under test:

### 1.1 Metrics for RAG System Evaluations
These metrics evaluate the core retrieval and generation components of a RAG pipeline:
* **Context Precision**: Evaluates whether all ground-truth relevant items in the retrieved context are ranked at the top.
* **Context Recall**: Measures if the retrieved context contains all necessary information to answer the question according to the ground truth.
* **Context Entity Recall**: Evaluates whether all key entities mentioned in the ground truth are present in retrieved contexts.
* **Noise Sensitivity**: Quantifies how often the generator makes errors when irrelevancies or noise are present in the context.
* **Answer Relevancy**: Assesses how directly and completely the generated answer addresses the user query.
* **Faithfulness**: Measures whether the generated answer is strictly grounded in the retrieved context (absence of hallucinations).
* **Factual Correctness**: Measures the factual overlap between the generated answer and the ground truth.
* **Semantic Similarity**: Calculates the semantic embedding similarity between the generated answer and ground truth.

```python
from ragas.metrics.collections import (
    ContextPrecision,
    ContextRecall,
    ContextEntityRecall,
    NoiseSensitivity,
    AnswerRelevancy,
    Faithfulness,
    FactualCorrectness,
    SemanticSimilarity,
)
```

### 1.2 Metrics for Multimodal RAG System Evaluations
Designed for RAG architectures incorporating vision, charts, diagrams, and images alongside text:
* **MultiModalFaithfulness**: Validates whether text generated from multimodal context (e.g., images, diagrams) faithfully reflects visual evidence.
* **MultiModalRelevance**: Assesses whether retrieved images and visual context are relevant to the query.

```python
from ragas.metrics.collections import (
    MultiModalFaithfulness,
    MultiModalRelevance,
)
```

### 1.3 Metrics for Agent & Tool Calling Evaluation
Specialized metrics for autonomous AI agents and tool-augmented workflows:
* **TopicAdherence**: Measures if an agent stays on topic across multi-turn interactions without drifting.
* **ToolCallAccuracy**: Evaluates whether the correct tool was selected and called with valid arguments.
* **ToolCallF1**: Measures the harmonic mean of precision and recall for tool selection.
* **AgentGoalAccuracyWithReference**: Evaluates if the agent satisfied user intent against a reference trajectory.
* **AgentGoalAccuracyWithoutReference**: Evaluates goal achievement purely from final state and conversational context.

```python
from ragas.metrics.collections import (
    TopicAdherence,
    ToolCallAccuracy,
    ToolCallF1,
    AgentGoalAccuracyWithReference,
    AgentGoalAccuracyWithoutReference,
)
```

### 1.4 Metrics for Text / Answer Comparison Evaluation (Natural Language Comparison)
Traditional and semantic NLP metrics for comparing generated output against ground-truth references:
* **FactualCorrectness**: LLM-driven factual entailment check.
* **SemanticSimilarity**: Cosine distance in embedding vector space.
* **NonLLMStringSimilarity**: Algorithmic string distance metrics without invoking an LLM.
* **BleuScore / RougeScore / CHRFScore**: Standard n-gram precision, recall, and character-level overlap metrics.
* **ExactMatch / StringPresence**: Deterministic substring and exact match verifications.

```python
from ragas.metrics.collections import (
    FactualCorrectness,
    SemanticSimilarity,
    NonLLMStringSimilarity,
    DistanceMeasure,
    BleuScore,
    CHRFScore,
    RougeScore,
    StringPresence,
    ExactMatch,
)
```

### 1.5 Primitive Metric Classes in Ragas
Core building blocks for creating custom evaluation metrics:
* **`discrete_metric` / `DiscreteMetric`**: For categorical outputs (e.g., Pass/Fail, Yes/No, Grade A/B/C).
* **`numeric_metric` / `NumericMetric`**: For bounded or continuous numeric scoring (e.g., 1–5 Likert scale, 0.0–1.0 scores).
* **`ranking_metric` / `RankingMetric`**: For evaluating sorted document rankings and ordering quality.

```python
from ragas.metrics import (
    discrete_metric,
    numeric_metric,
    ranking_metric,
)
```

### 1.6 Rubric Collections in Ragas
Standardized rubrics for domain-level or instance-specific criteria:
* **`DomainSpecificRubrics`**: Predefined evaluation guidelines tailored for specific vertical industries (e.g., legal, finance, healthcare).
* **`InstanceSpecificRubrics`**: Dynamic rubrics tailored to individual questions or evaluation samples.

```python
from ragas.metrics.collections import (
    DomainSpecificRubrics,
    InstanceSpecificRubrics,
)
```

## 2. End-to-End System Architecture

The system consists of two decoupled pipelines:
1. **LangChain RAG Pipeline**: Embeds documents using `GoogleGenerativeAIEmbeddings`, indexes them in `InMemoryVectorStore`, retrieves top-k chunks, formats them into a strictly grounded prompt, and generates answers using `ChatGoogleGenerativeAI` (`gemini-3.1-flash-lite`).
2. **Ragas Evaluation Engine**: An automated testing harness where an independent Evaluator LLM (`gemini-3-flash-preview`) systematically assesses each generated response against a ground-truth dataset across discrete (Correctness) and numeric (Relevance) metrics.

```text
What we building in this Notebook:

                                      RAG
                                       │
              ┌────────────────────────┼────────────────────────┐
              │                        │                        │
GoogleGenerativeAIEmbeddings      Vector Store              Retriever
                                                                │
                                                                ▼
                                                         Retrieved Docs
                                                                │
                                                                ▼
                                                       ChatPromptTemplate
                                                                │
                                                                ▼
                                                     ChatGoogleGenerativeAI
                                                                │
                                                                ▼
                                                         StrOutputParser
                                                                │
                                                                ▼
                                                            Response
                                                                │
                                                                │
                                  RAGAS EVALUATION              │
                                       │                        │
                                       ▼                        ▼
                                    Dataset                  Response
                                       │                        │
                                       └───────────┬────────────┘
                                                   ▼
                                              @experiment
                                                   │
                                  ┌────────────────┴───────────────┐
                                  │                                │
                            DiscreteMetric                   NumericMetric
                             Correctness                      Relevance
                                  │                                │
                                  └────────────────┬───────────────┘
                                                   ▼
                                             Score + Reason
                                                   │
                                                   ▼
                                           Experiment Results
                                                   │
                                                   ▼
                                                  CSV

```

## 3. Evaluation Theory: Rubric vs. Likert Scale

When designing automated LLM-as-a-Judge evaluation metrics, two primary measurement systems are used:

| Dimension | Rubric-Based Evaluation (`DiscreteMetric`) | Likert Scale Evaluation (`NumericMetric`) |
| :--- | :--- | :--- |
| **Definition** | Explicit, rule-based criteria evaluated categorically (e.g., `PASS` / `FAIL`, `Accurate` / `Contradictory`). | An ordinal, graded rating scale (e.g., 1 to 5) evaluating degree or intensity. |
| **Output Type** | Discrete labels / classifications (`allowed_values=["pass", "fail"]`). | Bounded numerical values (`allowed_values=(1, 5)`). |
| **Primary Use Cases** | **Factual Correctness, Safety, Policy Compliance, Binary Requirements**: Did the model follow rule X? Is the answer truthful? | **Relevancy, Fluency, Helpfulness, Tone, Comprehensiveness**: To what extent is the response helpful or relevant? |
| **Judge Consistency** | Very High. Clear objective boundaries leave little room for LLM subjective drift. | Moderate. Requires clear rubric anchors for each number to prevent judge variance. |
| **Actionability** | Immediate: A "FAIL" pinpoints an exact rule violation. | Statistical: Enables calculating means, medians, and distribution trends over time. |

---

### 💡 Key Design Rules for LLM-as-a-Judge:
1. **Explicit Scoring Anchors**: For numeric scales, define explicitly what constitutes a score of 1, 2, 3, 4, and 5 (never let the judge invent definitions).
2. **Reasoning First (`reason`)**: Asking the judge to state its reasoning before outputting its score forces chain-of-thought analysis and drastically improves reliability.
3. **Model Decoupling**: Never use the same model instance for generation and evaluation. We use `gemini-3.1-flash-lite` for generation and `gemini-3-flash-preview` for judgment.
4. **Strict Output Bounding**: Constrain `allowed_values` so downstream data pipelines receive predictable, error-free types.

---
## 4. Environment Setup & API Configuration

In this section, we load environment variables from `.env` and verify API key availability.
* We configure **`GOOGLE_API_KEY`** (with fallback to `GEMINI_API_KEY`) and set both in `os.environ` so all downstream Google GenAI and LangChain packages discover it automatically.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY or GEMINI_API_KEY not found in .env")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GEMINI_API_KEY"] = GOOGLE_API_KEY

print("API key loaded successfully")

API key loaded successfully


### 4.1 Model Hyperparameters & Configuration Constants
We establish explicit role separation for models:
* **`MODEL = 'gemini-3.1-flash-lite'`**: High-throughput, low-latency generator model for the RAG chain.
* **`EMBEDDING_MODEL = 'gemini-embedding-001'`**: Google's text embedding model producing dense 3072-dimensional vector representations.
* **`JUDGE_MODEL = 'gemini-3-flash-preview'`**: Advanced reasoning model serving as the impartial evaluation judge.
* **`TOP_K = 2`**: Number of context chunks retrieved for each query.

In [3]:
MODEL = "gemini-3.1-flash-lite"

EMBEDDING_MODEL = "gemini-embedding-001"

JUDGE_MODEL = "gemini-3-flash-preview"

TOP_K = 2

---
## 5. LangChain Google GenAI Components Initialization

We import and initialize the core LangChain Google GenAI classes:
* **`ChatGoogleGenerativeAI`**: The conversational chat model interface. Setting `temperature=0` ensures deterministic, grounded generation.
* **`GoogleGenerativeAIEmbeddings`**: Converts document strings and search queries into dense vector embeddings.

In [4]:
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)

from langchain_core.documents import Document

from langchain_core.prompts import (
    ChatPromptTemplate,
)

from langchain_core.output_parsers import (
    StrOutputParser,
)

from langchain_core.vectorstores import (
    InMemoryVectorStore,
)

In [5]:
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    temperature=0,
)

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


### 5.1 Sanity Check: Model Invocation
A quick test invocation to verify API connectivity and response generation before building the pipeline.

In [6]:
llm.invoke("hi how are you?")

AIMessage(content=[{'type': 'text', 'text': "I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EnMKcQFpFH0T6L3VWGD8kMC0pUzfZz/msw0Vtp63XtTvG6MqlG28q+z1dEWjxRPPgcW0AjYrcqQ1yaTUvsWUfeJ7g99Im+93zjLJ4buVVkhTZM7s11axb8QBYFtzP75+CJlir3FH4FKSq7+wGg9pno0zgJo4'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e541-f6ce-7b51-8e74-184ec4bce7ce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 26, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}})

---
## 6. Knowledge Base Construction & Vector Store Indexing

We define a curated synthetic knowledge base about **Ragas** and **RAG systems**.
Each document is wrapped in a LangChain `Document` object with:
* **`page_content`**: The raw text that will be embedded and retrieved.
* **`metadata` (`doc_id`)**: Unique identifier for tracking document provenance and retrieval accuracy.

In [7]:
# ============================================================
# 6. KNOWLEDGE BASE
# ============================================================
DOCUMENTS = [

    Document(
        page_content="""
Ragas is an open-source evaluation framework designed
for evaluating and improving LLM applications.

It supports RAG evaluation, agent evaluation,
custom metrics, datasets, experiments,
and synthetic test data generation.
""".strip(),
        metadata={
            "doc_id": "ragas_intro",
        },
    ),

    Document(
        page_content="""
Ragas experiments allow developers to systematically
evaluate AI applications, track evaluation results,
compare different versions, and store experiment outputs.
""".strip(),
        metadata={
            "doc_id": "ragas_experiments",
        },
    ),

    Document(
        page_content="""
Ragas metrics measure different properties of AI systems.

Metrics can evaluate factual correctness,
faithfulness, answer relevancy,
retrieval quality, agent behavior,
and application-specific requirements.
""".strip(),
        metadata={
            "doc_id": "ragas_metrics",
        },
    ),

    Document(
        page_content="""
RAG stands for Retrieval-Augmented Generation.

A RAG system retrieves relevant information
from a knowledge base and provides that information
to an LLM for answer generation.
""".strip(),
        metadata={
            "doc_id": "rag_intro",
        },
    ),

]

print(
    "\nKnowledge Base Documents:",
    len(DOCUMENTS),
)


Knowledge Base Documents: 4


### 6.1 In-Memory Vector Store & Document Indexing
We index `DOCUMENTS` into an **`InMemoryVectorStore`** using `GoogleGenerativeAIEmbeddings`.
* The vector store computes vector embeddings for each document chunk and stores them in memory.
* Ideal for local experimentation, evaluation benchmarks, and rapid unit testing without external database infrastructure.

In [8]:
vector_store = InMemoryVectorStore(embedding=embeddings,)

vector_store.add_documents(DOCUMENTS)

['598371a2-5298-41d1-a482-9bb904831fee',
 'c53ffb17-c104-4c6a-ab59-56947cc037e5',
 '0e18f616-a45e-4fe7-b39d-41a41da050ed',
 '618fb6e0-032d-47de-9dee-f25ef4a675f5']

### 6.2 Retriever Construction
We convert the vector store into a LangChain `Retriever` configured with `search_kwargs={'k': TOP_K}` to return the top 2 most semantically relevant chunks for any given query.

In [9]:
retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K,})

---
## 7. Grounded Prompt Engineering & LCEL RAG Chain

A robust RAG system requires strict prompt engineering to prevent hallucinations:
1. **Context-Only Grounding**: Instruct the model to use *only* the supplied context.
2. **Explicit Refusal Protocol**: If the answer is absent from context, the model must output:
   *`"I don't know based on the provided context."`*
3. **No Outside Knowledge**: Prohibit external parametric knowledge from bleeding into answers.

In [10]:
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the user's question ONLY using
the supplied context.

Rules:

1. Use only the provided context.
2. Do not invent information.
3. Do not use outside knowledge.
4. If the answer is not present in the context,
   say:

   "I don't know based on the provided context."

5. Keep the answer concise and factual.


CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""
)

### 7.1 Helper: Context Document Formatter
A utility function to join retrieved document chunks into a clean, double-newline-separated context block.

In [11]:
def format_documents(documents,):
    return "\n\n".join(document.page_content for document in documents)

### 7.2 LangChain Expression Language (LCEL) Chain Construction
We assemble the RAG generation chain using LCEL pipe (`|`) syntax:
$$\text{Prompt Template} \longrightarrow \text{ChatGoogleGenerativeAI (LLM)} \longrightarrow \text{StrOutputParser}$$
This creates a runnable that natively supports both synchronous (`invoke`) and asynchronous (`ainvoke`) execution.

In [12]:
rag_chain = prompt | llm | StrOutputParser()

---
## 8. Complete LangChain RAG Query Application

We encapsulate the complete RAG query flow in an asynchronous function `rag_query(question)`:
* **Step 1 (Retrieval)**: Asynchronously retrieve relevant documents via `retriever.ainvoke(question)`.
* **Step 2 (Context Assembly)**: Extract document IDs and format text chunks into a unified context block.
* **Step 3 (Answer Generation)**: Invoke `rag_chain.ainvoke()` to generate the grounded response.
* **Return Value**: A structured dictionary containing:
  - `answer`: The generated answer string.
  - `retrieved_contexts`: List of raw text chunks used for context.
  - `retrieved_document_ids`: List of document IDs for provenance tracking.

In [13]:
# ============================================================
# 12. COMPLETE LANGCHAIN RAG APPLICATION
# ============================================================

async def rag_query( question: str):
    
    # --------------------------------------------------------
    # STEP 1
    # Retrieve relevant documents using LangChain Retriever
    # --------------------------------------------------------
    relevant_documents = await retriever.ainvoke(question)
    retrieved_contexts = [ document.page_content for document in relevant_documents]
    retrieved_document_ids = [document.metadata.get("doc_id") for document in relevant_documents]

    # --------------------------------------------------------
    # STEP 2
    # Create context
    # --------------------------------------------------------
    context = format_documents(relevant_documents)

    # --------------------------------------------------------
    # STEP 3
    # Generate answer using LangChain LCEL chain
    # --------------------------------------------------------
    answer = await rag_chain.ainvoke(
        {
            "context": context,
            "question": question,
        }
    )

    return {
        "answer":answer,
        "retrieved_contexts": retrieved_contexts,
        "retrieved_document_ids": retrieved_document_ids
    }

---
## 9. Evaluation Setup & Golden Ground-Truth Dataset

To evaluate a RAG system systematically, we need a **Golden Dataset** (ground-truth dataset).
A golden dataset provides:
* **`question`**: Test user queries representing typical or edge-case usage.
* **`expected_answer`**: Verified, human-curated ground-truth answers.

We use Ragas's **`Dataset`** abstraction with a local CSV backend (`evals/ragas_langchain_dataset.csv`) to manage and persist test samples.

In [14]:
import asyncio
from pathlib import Path
import pandas as pd

In [21]:
from ragas import (
    Dataset,
    experiment,
)

from ragas.llms import (
    llm_factory,
)

from ragas.metrics import (
    DiscreteMetric,
    NumericMetric,
)

In [22]:
def load_dataset():
    Path("evals").mkdir(exist_ok=True)

    dataset = Dataset(
        name="ragas_langchain_dataset",
        backend="local/csv",
        root_dir="evals",
    )

    # golden dataset(ground truth)
    data_samples = [

        {
            "question": "What is Ragas?",

            "expected_answer": """
Ragas is an evaluation framework
for LLM applications.
""".strip(),
        },

        {
            "question": "How do Ragas experiments work?",

            "expected_answer": """
Ragas experiments provide systematic evaluation,
track results, compare different application versions,
and store experiment outputs.
""".strip(),
        },

        {
            "question": "What do Ragas metrics do?",

            "expected_answer": """
Ragas metrics evaluate properties and quality
of AI or LLM application outputs.
""".strip(),
        },

        {
            "question": "What is RAG?",

            "expected_answer": """
RAG stands for Retrieval-Augmented Generation.

It retrieves relevant information from a knowledge
source and provides that context to an LLM
before answer generation.
""".strip(),
        },

    ]

    for sample in data_samples:
        dataset.append(sample)

    dataset.save()

    return dataset

---
## 10. Google GenAI Evaluator Client & Judge LLM Configuration

To achieve objective, unbiased evaluation, we decouple the Evaluator LLM from the generation model:
* **Client**: Modern `google.genai.Client` initialized with `GOOGLE_API_KEY`.
* **Evaluator LLM**: Initialized via Ragas **`llm_factory`** with `provider='google'` using `gemini-3-flash-preview`.
  - `llm_factory` automatically configures structured output handling, enabling the judge to return structured Pydantic models containing both verdicts and reasoning.

In [17]:
from google import genai

In [18]:
ragas_gemini_client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"],
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [19]:
evaluator_llm = llm_factory(
    JUDGE_MODEL,
    provider="google",
    client=ragas_gemini_client,
)

---
## 11. Custom Metric Engineering: Discrete & Numeric Metrics

We implement two distinct evaluation metrics following the Rubric vs. Likert paradigms:

### 11.1 Discrete Metric: Factual Correctness (Pass / Fail Rubric)
* **Goal**: Determine whether the generated response contains all essential facts from the expected ground-truth answer.
* **Type**: `DiscreteMetric` with `allowed_values=['pass', 'fail']`.
* **Evaluation Prompt**: Explicit rules for when to award `pass` (factually correct, contains key info, answers query) vs `fail` (contradicts, missing info, irrelevant).

In [23]:
correctness_metric = DiscreteMetric(

    name="correctness",

    prompt="""
Evaluate whether the model response correctly answers
the user's question according to the expected answer.

QUESTION:

{question}


EXPECTED ANSWER:

{expected_answer}


MODEL RESPONSE:

{response}


Evaluation Rules:

PASS if:

1. The response contains the important information
   from the expected answer.

2. The response is factually correct.

3. The response answers the user's question.


FAIL if:

1. Important information is missing.

2. The response contradicts the expected answer.

3. The response does not answer the question.


Return only:

pass

or

fail
""",

    allowed_values=[
        "pass",
        "fail",
    ],

)

### 11.2 Numeric Metric: Answer Relevance (5-Point Likert Scale)
* **Goal**: Measure how directly and appropriately the response answers the user's specific question.
* **Type**: `NumericMetric` with `allowed_values=(1, 5)`.
* **5-Point Rubric Anchors**:
  - `1`: Completely irrelevant
  - `2`: Mostly irrelevant
  - `3`: Partially relevant
  - `4`: Mostly relevant
  - `5`: Completely relevant

In [24]:
relevance_metric = NumericMetric(

    name="relevance",

    prompt="""
Evaluate how relevant the model response is
to the user's question.


QUESTION:

{question}


MODEL RESPONSE:

{response}


Use this rubric:

1 = Completely irrelevant

2 = Mostly irrelevant

3 = Partially relevant

4 = Mostly relevant

5 = Completely relevant


Return only the numerical score.
""",

    allowed_values=(
        1,
        5,
    ),

)

---
## 12. Ragas Experiment Execution & Asynchronous Evaluation Loop

The **`@experiment()`** decorator transforms an async evaluation function into a structured, trackable test runner:
1. Iterates through each sample in the dataset.
2. Invokes the live RAG pipeline (`rag_query`) to obtain actual runtime outputs and retrieval contexts.
3. Evaluates output asynchronously using `correctness_metric.ascore()` and `relevance_metric.ascore()`.
4. Captures metric values (`.value`) and detailed rationale explanations (`.reason`) for comprehensive diagnostic analysis.

In [25]:
@experiment()
async def run_rag_evaluation(row):

    # --------------------------------------------------------
    # Get question
    # --------------------------------------------------------
    question = row["question"]

    # --------------------------------------------------------
    # Get expected answer
    # --------------------------------------------------------
    expected_answer = row["expected_answer"]


    # ========================================================
    # RUN LANGCHAIN RAG
    # ========================================================

    rag_result = await rag_query(question)
    response = rag_result["answer"]
    retrieved_contexts = rag_result["retrieved_contexts"]
    retrieved_document_ids = rag_result["retrieved_document_ids"]


    # ========================================================
    # RAGAS CORRECTNESS EVALUATION
    # ========================================================

    correctness_result = await correctness_metric.ascore(
        llm=evaluator_llm,
        question=question,
        expected_answer=expected_answer,
        response=response,
    )


    # ========================================================
    # RAGAS RELEVANCE EVALUATION
    # ========================================================

    relevance_result = await relevance_metric.ascore(
        llm=evaluator_llm,
        question=question,
        response=response,
    )


    # ========================================================
    # RETURN EXPERIMENT DATA
    # ========================================================

    return {
        **row,

        # Application Output
        "response": response,

        # Retrieval Information
        "retrieved_contexts": retrieved_contexts,
        "retrieved_document_ids": retrieved_document_ids,

        # Correctness
        "correctness": correctness_result.value,
        "correctness_reason": correctness_result.reason,

        # Relevance
        "relevance": relevance_result.value,
        "relevance_reason": relevance_result.reason,
    }

### 12.1 Experiment Orchestration & Reporting (`main`)
The `main()` function handles the complete evaluation workflow:
1. **Runs the experiment**: `await run_rag_evaluation.arun(dataset)`.
2. **Tabulates results**: Converts evaluation outputs to a pandas `DataFrame`.
3. **Calculates aggregate statistics**: Computes **Correctness Pass Rate (%)** and **Average Relevance Score (/5)**.
4. **Persists experiment results**: Exports the full report to `evals/experiments/ragas_langchain_results.csv`.
5. **Inspects qualitative reasons**: Prints individual question/response pairs along with the judge's reasoning for total transparency.

In [26]:
async def main():

    print("\n" + "=" * 100)
    print("RAGAS + LANGCHAIN EVALUATION")
    print("=" * 100)

    # ========================================================
    # LOAD DATASET
    # ========================================================

    dataset = load_dataset()

    print("\nDataset loaded.")

    # ========================================================
    # RUN RAGAS EXPERIMENT
    # ========================================================

    results = await run_rag_evaluation.arun(
        dataset,
        name="ragas_langchain_experiment",
    )

    # ========================================================
    # CONVERT RESULTS TO PANDAS
    # ========================================================

    df = results.to_pandas()

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    columns_to_show = [
        "question",
        "response",
        "correctness",
        "relevance",
    ]

    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # CORRECTNESS PASS RATE
    # ========================================================

    correctness_pass_rate = (df["correctness"].astype(str).str.lower().eq("pass").mean())

    # ========================================================
    # AVERAGE RELEVANCE
    # ========================================================

    average_relevance = pd.to_numeric(df["relevance"],errors="coerce",).mean()

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    print(f"Correctness Pass Rate : {correctness_pass_rate:.1%}")
    print(f"Average Relevance     : {average_relevance:.2f}/5")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    experiment_directory = Path("evals/experiments")
    experiment_directory.mkdir(parents=True, exist_ok=True)

    output_file = experiment_directory / "ragas_langchain_results.csv"

    df.to_csv(output_file, index=False)

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT DETAILED REASONS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED EVALUATION REASONS")
    print("=" * 100)

    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nRESPONSE:")
        print(result_row["response"])

        print("\nCORRECTNESS:")
        print(result_row["correctness"])

        print("\nCORRECTNESS REASON:")
        print(result_row["correctness_reason"])

        print("\nRELEVANCE:")
        print(result_row["relevance"])

        print("\nRELEVANCE REASON:")
        print(result_row["relevance_reason"])

        print("\nRETRIEVED DOCUMENT IDS:")
        print(result_row["retrieved_document_ids"])

        print("\n" + "-" * 100)

### 12.2 Async Event Loop Runner
Executes `main()` with automatic fallback to `nest_asyncio` for interactive Jupyter notebook kernels where an event loop is already active.

In [27]:
try:
    asyncio.run(main())

except RuntimeError as error:

    # Jupyter Notebook fallback
    if "asyncio.run() cannot be called" in str(error) or "event loop" in str(error).lower():
        import nest_asyncio
        nest_asyncio.apply()

        loop = asyncio.get_event_loop()
        loop.run_until_complete(main())

    else:
        raise


RAGAS + LANGCHAIN EVALUATION

Dataset loaded.


Running experiment: 100%|██████████| 4/4 [06:49<00:00, 102.30s/it]


DETAILED RESULTS
                      question                                                                                                                                                                               response correctness  relevance
                  What is RAG? RAG stands for Retrieval-Augmented Generation. It is a system that retrieves relevant information from a knowledge base and provides that information to an LLM for answer generation.        pass        5.0
How do Ragas experiments work?                     Ragas experiments allow developers to systematically evaluate AI applications, track evaluation results, compare different versions, and store experiment outputs.        pass        4.0
     What do Ragas metrics do?                                                                                                                              Ragas metrics measure different properties of AI systems.        pass        5.0
                What is Ragas?    


C:\Users\ahmad\AppData\Local\Temp\ipykernel_20524\3836954189.py:12: RuntimeWarning: coroutine 'main' was never awaited
  loop.run_until_complete(main())
